# RK1828 Qwen3-VL Vision-Language Model Demo


[Qwen3-VL](https://github.com/QwenLM/Qwen3-VL) (Tongyi Qianwen): image QA /
captioning. The LLM part is **GRQ W4A16 quantized**, the vision encoder is
fp16, both run on the RK1828.

| Variant | Generation speed (measured) |
|---|---|
| Qwen3-VL-2B | 62.7 tokens/s |
| Qwen3-VL-4B | 38.7 tokens/s |

| Step | Content |
|---|---|
| 1 | Download models (from huggingface.co; you may fetch only 2B or 4B) |
| 2 | Extract the prebuilt demo |
| 3 | Enter a prompt and chat with the image |

## 0. Environment check

Make sure the RKNN3 runtime (`librknn3_api.so`) and the RK1828 device are ready.
If this fails, install the driver/runtime first (the `rknn3-rk182x` DEB package)
and check with `sudo rknn-smi info` that the RK1828 shows up.

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # loads /usr/lib/librknn3_api.so; failure means the runtime is not installed

devices = find_devices()
if not devices:
    raise RuntimeError("No RK1828 device found! Check: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"Found device: id={dev_id}  interface={dev_type}")

## 1. Download models

The models for this demo are hosted on Hugging Face (account `qhddz`) and are
downloaded directly from **huggingface.co**. Run the next cell to start.

> Files that already exist (and are non-empty) are **skipped automatically** —
> re-running never downloads them twice.
> If your network cannot reach huggingface.co, set up a proxy first, e.g.
> `export https_proxy=http://<your-proxy>:<port>` before starting Jupyter.


Both variants total ~10GB. If disk is tight, edit the download cell to fetch only one variant.

In [ ]:
# (HF repo, path in repo, local path)
FILES = [
    ("qwen3-vl-2b-rknn3-rk1828", "Qwen3-VL-2B-llm.embed.bin", "demos/qwen3_vl_2b/Qwen3-VL-2B-llm.embed.bin"),
    ("qwen3-vl-2b-rknn3-rk1828", "Qwen3-VL-2B-llm.rknn", "demos/qwen3_vl_2b/Qwen3-VL-2B-llm.rknn"),
    ("qwen3-vl-2b-rknn3-rk1828", "Qwen3-VL-2B-llm.tokenizer.gguf", "demos/qwen3_vl_2b/Qwen3-VL-2B-llm.tokenizer.gguf"),
    ("qwen3-vl-2b-rknn3-rk1828", "Qwen3-VL-2B-llm.weight", "demos/qwen3_vl_2b/Qwen3-VL-2B-llm.weight"),
    ("qwen3-vl-2b-rknn3-rk1828", "Qwen3-VL-2B-vision.rknn", "demos/qwen3_vl_2b/Qwen3-VL-2B-vision.rknn"),
    ("qwen3-vl-2b-rknn3-rk1828", "Qwen3-VL-2B-vision.weight", "demos/qwen3_vl_2b/Qwen3-VL-2B-vision.weight"),
    ("qwen3-vl-2b-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/qwen3_vl_2b/demo_x86_64.tar.gz"),
    ("qwen3-vl-2b-rknn3-rk1828", "model/demo.jpg", "demos/qwen3_vl_2b/model/demo.jpg"),
    ("qwen3-vl-4b-rknn3-rk1828", "Qwen3-VL-4B-llm.embed.bin", "demos/qwen3_vl_4b/Qwen3-VL-4B-llm.embed.bin"),
    ("qwen3-vl-4b-rknn3-rk1828", "Qwen3-VL-4B-llm.rknn", "demos/qwen3_vl_4b/Qwen3-VL-4B-llm.rknn"),
    ("qwen3-vl-4b-rknn3-rk1828", "Qwen3-VL-4B-llm.tokenizer.gguf", "demos/qwen3_vl_4b/Qwen3-VL-4B-llm.tokenizer.gguf"),
    ("qwen3-vl-4b-rknn3-rk1828", "Qwen3-VL-4B-llm.weight", "demos/qwen3_vl_4b/Qwen3-VL-4B-llm.weight"),
    ("qwen3-vl-4b-rknn3-rk1828", "Qwen3-VL-4B-vision.rknn", "demos/qwen3_vl_4b/Qwen3-VL-4B-vision.rknn"),
    ("qwen3-vl-4b-rknn3-rk1828", "Qwen3-VL-4B-vision.weight", "demos/qwen3_vl_4b/Qwen3-VL-4B-vision.weight"),
    ("qwen3-vl-4b-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/qwen3_vl_4b/demo_x86_64.tar.gz"),
    ("qwen3-vl-4b-rknn3-rk1828", "model/demo.jpg", "demos/qwen3_vl_4b/model/demo.jpg"),
]

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, "hf")

## 2. Extract the prebuilt demo

`demo_x86_64.tar.gz` contains the official x86_64 C++ demo and its runtime
libraries (`lib/`). Hugging Face does not host raw executables, so they are
shipped as a tarball; extract once (re-running skips automatically).

> **Why a C++ demo instead of pure Python?**
> Notebooks 01-05 (classification / detection / FastSAM / pose / segmentation)
> are a single model with one forward pass and simple post-processing, so they
> walk through everything in Python (`rknn3.py`). This demo's pipeline is far
> more complex — chained models, tokenizer, KV cache, sliding-window /
> streaming state, hundreds of lines of orchestration — reimplementing it in
> Python would be costly and risk diverging from the official behavior.
> So we run the official `rknn3-model-zoo` C++ demo directly; source code:
> [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> under `examples/`.

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/qwen3_vl_2b/demo_x86_64.tar.gz", "demos/qwen3_vl_2b")
extract_tarball("demos/qwen3_vl_4b/demo_x86_64.tar.gz", "demos/qwen3_vl_4b")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. Pick a model variant and a prompt

In [ ]:
import ipywidgets as widgets
from IPython.display import display

w_model = widgets.Dropdown(options=[("Qwen3-VL-2B (faster)", "2b"),
                                    ("Qwen3-VL-4B (smarter)", "4b")],
                           value="2b", description="Model:")
w_prompt = widgets.Textarea(value="Describe this image",
                            description="Prompt:",
                            layout=widgets.Layout(width="70%", height="60px"))
display(w_model, w_prompt)
print("Then keep running downwards ▼")

In [ ]:
DEMO = f"demos/qwen3_vl_{w_model.value}"
tag = "Qwen3-VL-2B" if w_model.value == "2b" else "Qwen3-VL-4B"
show_images([f"{DEMO}/model/demo.jpg"], max_width=400)
run_demo(["./rknn_qwen3_vl_demo",
          f"{tag}-vision.rknn", f"{tag}-vision.weight",
          f"{tag}-llm.rknn", f"{tag}-llm.weight",
          f"{tag}-llm.tokenizer.gguf", f"{tag}-llm.embed.bin",
          "0xff", "0xff", "model/demo.jpg", w_prompt.value, "384", "384"], DEMO)

## Summary

- A VLM = vision encoder (image -> token embeddings) + LLM (autoregressive
  text); both are separate RKNN models.
- `0xff 0xff` are device/session placeholders; `384 384` is the vision input
  resolution.
- Try Chinese questions, counting people in the image, OCR, etc.